# Project 01: Stock Screener: Rule-Based Screening & Rolling Evaluation
**Objective:** Expand the asset universe and implement a rule-based filtering logic based on volatility, momentum, and drawdown metrics, evaluated on a rolling basis.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import duckdb
import yfinance as yf
import plotly.express as px

In [2]:
# Extraction and Normalization of Data
tickers = ["NVDA", "MSFT", "MSTR", "JPM", "BAC", "MS", "LLY", "JNJ", "ABBV", "XOM", "CVX", "COP", "PG", "KO", "PEP", "SPY"]
start_date = "2022-06-01"
end_date = "2025-01-01"

market_data_wide = yf.download(tickers, start_date, end_date, auto_adjust=True, progress=False, multi_level_index=True)
market_data = market_data_wide.stack().reset_index()
market_data.columns = market_data.columns.str.lower()
market_data.columns.name = None
market_data = market_data[["date", "ticker", "open", "high", "low", "close", "volume"]]

# Generation of Analytical Base Table in-memory
query = """
WITH previous_prices AS (
    SELECT 
        date, ticker, open, high, low, close, volume,
        LAG(close) OVER (PARTITION BY ticker ORDER BY date) AS previous_close
    FROM market_data
)
SELECT 
    date, ticker, open, high, low, close, volume,
    (close / previous_close) - 1 AS daily_return
FROM previous_prices
ORDER BY ticker, date;
"""

daily_returns = duckdb.sql(query).df()

# Validation
assert daily_returns.shape[1] == 8, "Expected 8 columns in the Analytical Base Table."
assert daily_returns['daily_return'].isna().sum() == 16, "Expected exactly 16 NaNs for daily_return at t_0."
assert daily_returns.duplicated(subset=['date', 'ticker']).sum() == 0, "Duplicate composite primary keys found."

### Universe Expansion Strategy
The universe has been expanded from 3 to 16 highly liquid assets spanning 5 major GICS sectors (Tech, Financials, Healthcare, Energy, Consumer Staples), plus the SPY benchmark. 
*Rationale:* A 16-asset universe provides sufficient cross-sectional breadth to evaluate the screening logic dynamically, avoiding trivial pass/fail outcomes, while maintaining high liquidity constraints to simulate a realistic investable universe.

*Note on Data Ingestion:* The time window starts in June 2022 to account for the 126-day rolling window "burn-in" period, ensuring valid feature generation starting from early 2023.

In [ ]:
screening_features = duckdb.sql("""
WITH lag_values AS (
    SELECT 
        date, ticker, close,

        LAG(close, 125) OVER (PARTITION BY ticker ORDER BY date) AS close_126d_ago,
        MAX(close) OVER (PARTITION BY ticker ORDER BY date ROWS BETWEEN 125 PRECEDING AND CURRENT ROW) AS peak_126d,
        STDDEV_SAMP(daily_return) OVER(PARTITION BY ticker ORDER BY date ROWS BETWEEN 62 PRECEDING AND CURRENT ROW) AS raw_vol_63d,

        COUNT(close) OVER (PARTITION BY ticker ORDER BY date ROWS BETWEEN 125 PRECEDING AND CURRENT ROW) AS count_126d,
        COUNT(daily_return) OVER (PARTITION BY ticker ORDER BY date ROWS BETWEEN 62 PRECEDING AND CURRENT ROW) AS count_63d,

    FROM daily_returns
)
SELECT date, ticker,
CASE WHEN count_126d = 126 THEN (close / close_126d_ago) - 1 ELSE NULL END AS momentum_126d,
CASE WHEN count_63d = 63 THEN raw_vol_63d * SQRT(252) ELSE NULL END AS rolling_vol_63d,
CASE WHEN count_126d = 126 THEN (close / peak_126d) - 1 ELSE NULL END AS current_dd_126d
FROM lag_values
ORDER BY ticker, date
""").df()

# NOTE: arbitrary numbers (16, 62, 125) are hardcoded to the current universe size / window lengths. See README "Known Limitations".
assert screening_features['rolling_vol_63d'].isna().groupby(screening_features['ticker']).sum().min() >= 62, "Cold start for volatility not handled correctly."
assert screening_features['momentum_126d'].isna().groupby(screening_features['ticker']).sum().min() >= 125, "Cold start for momentum not handled correctly."
assert screening_features['current_dd_126d'].isna().groupby(screening_features['ticker']).sum().min() >= 125, "Cold start for MDD not handled correctly."
assert (screening_features['current_dd_126d'].dropna() <= 0).all(), "Max Drawdown cannot be positive."

# Screening Features
Three important features for the Stock Screener have been computed:

**1.** momentum_126d: calculate the return of a specific asset on a window of approximately six months (126 business days)

**2.** rolling_vol_63d: calculate the annualized volatility on a window of approximately three months (63 business days)

**3.** current_dd_126d: calculate the drawdown of the current value of the asset with respect to the maximum registered on a window of approximately six months

In [4]:
screener_results = duckdb.sql("""
WITH cross_sectional_features AS (
SELECT *,
QUANTILE_CONT(rolling_vol_63d, 0.5 ORDER BY rolling_vol_63d) OVER (PARTITION BY date) AS median
FROM screening_features
ORDER BY date, ticker
)
SELECT *,
momentum_126d > 0 AS pass_momentum,
rolling_vol_63d < median AS pass_volatility,
current_dd_126d >= -0.15 AS pass_drawdown,
COALESCE (momentum_126d > 0 AND rolling_vol_63d < median AND current_dd_126d >= -0.15, FALSE) AS is_investable
FROM cross_sectional_features
ORDER BY ticker, date
""").df()

# Total protection from cold-start: no NULL values may be investable
assert screener_results.loc[screener_results['momentum_126d'].isna(), 'is_investable'].sum() == 0, "Leakage: Assets in burn-in period passed the screener."

# Formal check of aggregate boolean logic
expected_investable = (
    screener_results['pass_momentum'] & 
    screener_results['pass_drawdown'] & 
    screener_results['pass_volatility']
).fillna(False)
assert (screener_results['is_investable'] == expected_investable).all(), "Boolean aggregation logic failed. Check NULL handling."

# Cross-sectional check: is_investable must actively filter (not all False, not all True)
print(f"Total historical data points: {len(screener_results)}")
print(f"Total investable signals generated: {screener_results['is_investable'].sum()}")
# Show the most recent positive signals
display(screener_results[screener_results['is_investable']].tail(10))

Total historical data points: 10400
Total investable signals generated: 3015


,date,ticker,momentum_126d,rolling_vol_63d,current_dd_126d,median,pass_momentum,pass_volatility,pass_drawdown,is_investable
10380,2024-12-03,XOM,0.057214,0.208467,-0.053706,0.233929,True,True,True,True
10381,2024-12-04,XOM,0.019099,0.216049,-0.080968,0.237516,True,True,True,True
10382,2024-12-05,XOM,0.034633,0.215947,-0.076947,0.234117,True,True,True,True
10383,2024-12-06,XOM,0.020738,0.212966,-0.086678,0.231716,True,True,True,True
10384,2024-12-09,XOM,0.022949,0.200091,-0.092066,0.228937,True,True,True,True
10385,2024-12-10,XOM,0.032276,0.199039,-0.093916,0.226895,True,True,True,True
10386,2024-12-11,XOM,0.033698,0.197748,-0.099947,0.227032,True,True,True,True
10387,2024-12-12,XOM,0.041577,0.197752,-0.100751,0.226619,True,True,True,True
10388,2024-12-13,XOM,0.039594,0.196570,-0.108632,0.227029,True,True,True,True
10389,2024-12-16,XOM,0.007879,0.199282,-0.127692,0.226971,True,True,True,True
